# 10 — Timelapse Corridor Visualization

Generate cinematic AIS timelapse visualizations that reveal shipping corridors
through cumulative dot accumulation — like long-exposure photography of
maritime traffic.

The output is a self-contained HTML file with Canvas 2D additive blending
over a MapLibre GL dark basemap. Dots glow and accumulate over time to
form luminous corridors wherever vessel traffic is dense.

Supports both single-panel and multi-panel (synchronized) layouts.

In [ ]:
from neptune_ais import Neptune
from neptune_ais.viz import generate_timelapse, TimelapsConfig

## 1. Download AIS data

For a striking timelapse, use multiple days of data. NOAA provides free
archival AIS data for US coastal waters. We download 7 days for the
Port of Los Angeles area.

In [ ]:
LA_BBOX = (-118.4, 33.55, -117.8, 33.9)

n = Neptune(
    ("2024-06-15", "2024-06-21"),
    sources=["noaa"],
    bbox=LA_BBOX,
    cache_dir="/tmp/neptune_timelapse_demo",
)
n.download()

In [ ]:
positions = n.positions().collect()
print(f"{len(positions):,} positions from {positions['mmsi'].n_unique():,} vessels")

## 2. Generate single-panel timelapse

Configure the timelapse with `TimelapsConfig`. Key parameters:

- **`dot_radius`** — size of each glow dot (default 1.0; use smaller for dense data)
- **`dot_alpha`** — brightness per dot; with additive blending, overlapping dots intensify (default 0.10; lower = finer corridors)
- **`speed`** — animation speed in time-bins per second (default 2)
- **`bin_interval_minutes`** — how many minutes of data per animation step (default 30)
- **`bloom`** — gaussian blur post-processing for the "luminous rivers" glow effect (default True)
- **`fade_factor`** — how much old dots fade per frame; 1.0 = no fade/full accumulation, 0.99 = gradual fade (default 1.0)

In [ ]:
config = TimelapsConfig(
    title="VESSEL MARKET - AIS TIMELAPSE",
    subtitle=f"PORT OF LOS ANGELES - {positions['mmsi'].n_unique():,} VESSELS",
    date_from="2024-06-15",
    date_to="2024-06-21",
    center_lat=33.72,
    center_lon=-118.22,
    zoom=12,
    speed=6,
)

# Optional: enrich vessel types from the vessels table.
vessels = n.vessels().collect()

output = generate_timelapse(
    positions,
    vessels=vessels if len(vessels) > 0 else None,
    config=config,
    output="la_timelapse.html",
    max_points=200_000,
)

import os
print(f"Timelapse: {output}")
print(f"File size: {os.path.getsize(output) / 1024:.0f} KB")

Open `la_timelapse.html` in any browser. The visualization includes:

- **Glowing dots** that accumulate over time with additive blending
- **Bloom post-processing** — dense corridors radiate soft light
- **Vessel type coloring** — cargo (blue), tanker (rose), passenger (teal), fishing (amber), tug (purple)
- **Live vessel count** — ticks up as new vessels appear
- **Timestamp** — shows the current time in the animation
- **Controls** — Play/Pause (Space), scrub slider, speed selector
- **Legend** — vessel type color key

## 3. Multi-panel timelapse

Compare multiple ports simultaneously with synchronized playback.
Each panel has its own map, data, and configuration, but they
advance in lockstep.

In [ ]:
# Example: comparing LA and Long Beach approaches
# (In practice, you'd use separate Neptune instances per region)
#
# panels = [
#     {
#         "positions": la_positions,
#         "config": TimelapsConfig(
#             center_lat=33.74, center_lon=-118.27, zoom=13,
#         ),
#         "label": "Port of Los Angeles",
#     },
#     {
#         "positions": lb_positions,
#         "config": TimelapsConfig(
#             center_lat=33.75, center_lon=-118.19, zoom=13,
#         ),
#         "label": "Port of Long Beach",
#     },
# ]
#
# generate_timelapse(
#     positions=la_positions,  # ignored when panels is set
#     config=TimelapsConfig(
#         title="LA / LONG BEACH PORT COMPARISON",
#         layout="vertical",  # stacked vertically like Kpler
#     ),
#     panels=panels,
#     output="multi_timelapse.html",
# )

## 4. Adapting for other regions

Change the bounding box and config to create timelapses for any region.
For best results:

- Use 7-30 days of data for prominent corridors
- Adjust `dot_alpha` (lower for denser data, higher for sparser)
- Set `zoom` to frame the shipping lanes
- Use `max_points=200_000` for reasonable file sizes (~4-5 MB)

In [ ]:
# Example: Puget Sound / Pacific Northwest
#
# PUGET_BBOX = (-123.5, 47.0, -122.0, 49.0)
# n = Neptune(
#     ("2024-06-01", "2024-06-30"),
#     sources=["noaa"],
#     bbox=PUGET_BBOX,
# )
# n.download()
# positions = n.positions().collect()
#
# generate_timelapse(
#     positions,
#     config=TimelapsConfig(
#         title="PUGET SOUND - AIS TIMELAPSE",
#         center_lat=48.0,
#         center_lon=-122.8,
#         zoom=9,
#         speed=8,
#     ),
#     output="puget_timelapse.html",
# )